# 06 · Hyperparameter tuning: Grid Search vs Random Search vs Optuna (RQ3)
Same 5-fold stratified CV and the same F1 objective for all three; only the search strategy changes. Budgets are reduced here so the notebook runs in minutes.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
print("project root:", ROOT)

In [ ]:
from ml.models.optimization import HyperparameterOptimizer
from ml.models import optimization as opt
from ml.training.evaluate import evaluate_model
s = joblib.load("ml/data/splits/dataset_splits.joblib")
X_train, y_train, X_test, y_test = s["X_train"], s["y_train"], s["X_test"], s["y_test"]
opt.RF_GRID = {"n_estimators": [100, 200], "max_depth": [10, 20, None]}     # 6 configs
optimizer = HyperparameterOptimizer(n_trials=8, cv_folds=5, random_state=42)
rows = []
for method, fn in [("GridSearchCV", optimizer.grid_search_random_forest), ("RandomizedSearchCV", lambda X, y: optimizer.random_search_random_forest(X, y, n_iter=6)), ("Optuna TPE", optimizer.optimize_random_forest)]:
    est, res = fn(X_train, y_train)
    ev = evaluate_model(est, X_test, y_test, method)
    rows.append({"method": method, "configs": res["n_trials"], "wall_s": res["wall_time_sec"], "best_cv_f1": round(res["best_cv_f1"], 4), "test_f1": ev["f1"], "best_params": res["best_params"]})
pd.DataFrame(rows)

In [ ]:
hist = optimizer.study_results_["optimized_random_forest"]["trials_history"]
vals = [h["value"] for h in hist]
plt.scatter(range(len(vals)), vals, color="#0072B2"); plt.step(range(len(vals)), np.maximum.accumulate(vals), where="post", color="#D55E00")
plt.xlabel("Optuna trial"); plt.ylabel("CV F1"); plt.title("TPE convergence"); plt.tight_layout()